In [11]:
# Setup
#  - Create a project named "Lab01" under r"C:\GEOG3440".
#  - Download Network Data Set, 2023, published by DMTI Spatial Inc. 
#    from Scholar’s Geoportal.
#  - Place the extracted folder under r"C:\GEOG3440"

In [1]:
import datetime
from typing import Literal

import arcpy

# Define the path to Lab01 gdb
lab01_gdb = r"C:\GEOG3440\Lab01\Lab01.gdb"

# Validate paths
if not arcpy.Exists(lab01_gdb):
    raise FileNotFoundError(f"Main geodatabase not found: {lab01_gdb}")

In [28]:
arcpy.env.workspace = lab01_gdb

# Import the GTFS datum
path_gtfs = r"C:\GEOG3440\GEOG3440_F24_Lab01_GuelphGTFS"
fd_streets_name = "Streets"
fd_streets = f"{lab01_gdb}\\{fd_streets_name}"

arcpy.management.CreateFeatureDataset(
    out_dataset_path=lab01_gdb, 
    out_name=fd_streets_name, 
    spatial_reference=arcpy.SpatialReference(26917)  # EPSG for NAD83 / UTM zone 17N
)

arcpy.transit.GTFSToPublicTransitDataModel(
    in_gtfs_folders=[path_gtfs], 
    target_feature_dataset=fd_streets, 
    # {interpolate}, 
    # {append}, 
    # {make_lve_shapes}
)

<Result 'C:\\GEOG3440\\Lab01\\Lab01.gdb\\Streets'>

In [29]:
# Define the path to DMTI 2023 gdb
external_gdb = r"C:\GEOG3440\DMTI_2023_CMCS_NetworkDataSet\data\DMTI_2023_CMCS_NetworkDataSet.gdb\Network"

# Validate paths
if not arcpy.Exists(external_gdb):
    raise FileNotFoundError(f"External geodatabase not found: {external_gdb}")

In [30]:
# List all feature classes in the external geodatabase
arcpy.env.workspace = external_gdb
feature_classes = arcpy.ListFeatureClasses()

for fc in feature_classes:
    print(fc)

RoadsNetwork
NetworkDataSet_Junctions
PedestrianNetworkRegion
RestrictionNetwork


In [31]:
# Import the RoadsNetwork feature class as a feature layer
fc_roads_network = r"C:\GEOG3440\DMTI_2023_CMCS_NetworkDataSet\data\DMTI_2023_CMCS_NetworkDataSet.gdb\Network\RoadsNetwork"
fl_roads_network = "RoadsNetwork"
arcpy.management.MakeFeatureLayer(
    in_features=fc_roads_network, 
    out_layer=fl_roads_network, 
    # {where_clause}, 
    # {workspace}, 
    # {field_info}
)

<Result 'RoadsNetwork'>

In [32]:
arcpy.env.workspace = fd_streets

# Filter the roads in Guelph
fl_roads_network_guelph_tmp = "RoadsNetworkGuelphTmp"
arcpy.management.SelectLayerByAttribute(
    in_layer_or_view=fl_roads_network, 
    selection_type="NEW_SELECTION", 
    where_clause="LEFT_MAF = 'GUELPH'", 
    # {invert_where_clause}
)

# Make a feature class from the selected layer
arcpy.management.CopyFeatures(
    in_features=fl_roads_network, 
    out_feature_class=fl_roads_network_guelph_tmp, 
    # {config_keyword}, 
    # {spatial_grid_1}, 
    # {spatial_grid_2}, 
    # {spatial_grid_3}
)

# Add a ROAD_CLASS field to Streets to use template
arcpy.management.AddField(
    in_table=fl_roads_network_guelph_tmp, 
    field_name="ROAD_CLASS", 
    field_type="SHORT", 
    # {field_precision}, 
    # {field_scale}, 
    # {field_length}, 
    # {field_alias}, 
    # {field_is_nullable}, 
    # {field_is_required}, 
    # {field_domain}
)

# Add a RestrictPedestrians field to Streets to use template
arcpy.management.AddField(
    in_table=fl_roads_network_guelph_tmp, 
    field_name="RestrictPedestrians", 
    field_type="TEXT", 
    # {field_precision}, 
    # {field_scale}, 
    # {field_length}, 
    # {field_alias}, 
    # {field_is_nullable}, 
    # {field_is_required}, 
    # {field_domain}
)
with arcpy.da.UpdateCursor(
    fl_roads_network_guelph_tmp, 
    ("STREET", "RestrictPedestrians")
) as cursor:
    for row in cursor:
        if row[0] == "HANLON PKY":
            row[1] = "Y" 
        else:
            row[1] = "N"
        cursor.updateRow(row)


# Change the coordinate system
fl_roads_network_guelph = "Streets"  # The feature class name has to be "Streets" to use template later
arcpy.management.Project(
    in_dataset=fl_roads_network_guelph_tmp, 
    out_dataset=fl_roads_network_guelph, 
    out_coor_system=arcpy.SpatialReference(26917),  # the EPSG/WKID for NAD 1983 / UTM Zone 17N
    # {transform_method}, 
    # {in_coor_system}, 
    # {preserve_shape}, 
    # {max_deviation}, 
    # {vertical}
)

# Delete the temporary feature class and layer
arcpy.Delete_management(fl_roads_network)
arcpy.Delete_management(f"{fd_streets}\\{fl_roads_network_guelph_tmp}")
arcpy.Delete_management(fl_roads_network_guelph_tmp)

# Recreate the feature layer
arcpy.management.MakeFeatureLayer(
    in_features=fl_roads_network_guelph, 
    out_layer=fl_roads_network_guelph, 
    # {where_clause}, 
    # {workspace}, 
    # {field_info}
)

<Result 'Streets'>

In [33]:
# Connect the transit stops to the streets
arcpy.env.workspace = lab01_gdb
arcpy.transit.ConnectPublicTransitDataModelToStreets(
    target_feature_dataset=fd_streets_name, 
    in_streets_features=fl_roads_network_guelph, 
    search_distance=375, # We need 375 meters to connect all the points
    expression="RestrictPedestrians <> 'Y'"
)

<Result 'C:\\GEOG3440\\Lab01\\Lab01.gdb\\Streets'>

In [34]:
# Create the network dataset from a template
arcpy.na.CreateNetworkDatasetFromTemplate(
    network_dataset_template=r"C:\GEOG3440\Network_Analyst_Pro_Tutorial\Network Analyst\Tutorial\PublicTransit\TransitNetworkTemplate.xml", 
    output_feature_dataset=fd_streets_name
)

<Result 'C:\\GEOG3440\\Lab01\\Lab01.gdb\\Streets\\TransitNetwork_ND'>

In [ ]:
# You have to modify the property here

In [35]:
# Build the network dataset
arcpy.env.workspace = fd_streets
arcpy.na.BuildNetwork(
    in_network_dataset="TransitNetwork_ND", 
    # {force_full_build}
)

<Result 'TransitNetwork_ND'>

In [22]:
# Create a shapefile for U of G University Center
# arcpy.env.workspace = lab01_gdb
# arcpy.management.CopyFeatures(
#     in_features=r"SA-toUofGUC-Wed1000\Facilities", 
#     out_feature_class="UofGUC", 
#     # {config_keyword}, 
#     # {spatial_grid_1}, 
#     # {spatial_grid_2}, 
#     # {spatial_grid_3}
# )
# arcpy.conversion.FeatureClassToShapefile(
#     Input_Features="UofGUC", 
#     Output_Folder=r"C:\GEOG3440\Lab01\Facilities"
# )

<Result 'C:\\GEOG3440\\Lab01\\Facilities'>

In [2]:
def run_service_area_analysis(
    network_data_source: str, 
    layer_name: str, 
    public_transit_time: arcpy.nax.TravelMode, 
    travel_direction: Literal["FROM_FACILITIES", "TO_FACILITIES"], 
    cutoffs: list[int], 
    time_of_day: datetime.datetime, 
    geometry_at_overlaps: Literal["OVERLAP", "DISSOLVE", "SPLIT"], 
    facilities: str, 
) -> None:
    # Create a new service area layer.
    sa_object = arcpy.na.MakeServiceAreaAnalysisLayer(
        network_data_source=network_data_source, 
        layer_name=layer_name, 
        travel_mode=public_transit_time, 
        travel_direction=travel_direction, 
        cutoffs=cutoffs, 
        time_of_day=time_of_day, 
        # {time_zone}, 
        output_type="POLYGONS", 
        # {polygon_detail}, 
        geometry_at_overlaps=geometry_at_overlaps, 
        geometry_at_cutoffs="DISKS", 
        # {polygon_trim_distance}, 
        # {exclude_sources_from_polygon_generation}, 
        # {accumulate_attributes}, 
        # {ignore_invalid_locations}
    )
    
    # Get the layer object from the result object. The service area layer can
    # now be referenced using the layer object.
    sa_layer_object = sa_object.getOutput(0)
    
    # Get the names of all the sublayers within the service area layer.
    sa_sublayer_names = arcpy.na.GetNAClassNames(sa_layer_object)

    # Stores the layer names that we will use later.
    facilities_layer_name = sa_sublayer_names["Facilities"]

    # Load the fire stations as facilities.
    arcpy.na.AddLocations(
        in_network_analysis_layer=sa_layer_object, 
        sub_layer=facilities_layer_name, 
        in_table=facilities, 
        # {field_mappings}, 
        # {search_tolerance}, 
        # {sort_field}, 
        # {search_criteria}, 
        # {match_type}, 
        # {append}, 
        # {snap_to_position_along_network}, 
        # {snap_offset}, 
        # {exclude_restricted_elements}, 
        # {search_query}, 
        # {allow_auto_relocate}
    )

    # Run service area analysis with facilities.
    arcpy.na.Solve(
        sa_layer_object, 
        # ignore_invalids="SKIP", 
        # terminate_on_solve_error="TERMINATE"
    )

    # Save the service area analysis result
    arcpy.management.SaveToLayerFile(
        in_layer=sa_layer_object,
        out_layer=f"{layer_name}.lyrx",
    )
    
    return None

In [36]:
# ### Part2's service area analysis ### #
arcpy.env.workspace = fd_streets

# Import facilities (Great American Ballpark).
arcpy.conversion.FeatureClassToGeodatabase(
    Input_Features=r"C:\GEOG3440\Lab01\Facilities\UofGUC.shp", 
    Output_Geodatabase=fd_streets
)

# Set the public transit time.
network_data_source="TransitNetwork_ND"
modes = arcpy.na.GetTravelModes(network_data_source)
public_transit_time = modes["Public transit time"] 

# Run service area analysis for 15, 30, 45 and 60 minutes.
for i in [15, 30, 45, 60]:
    run_service_area_analysis(
        network_data_source=network_data_source, 
        layer_name=f"SA-toUofGUC-Wed1000-{str(i)}min", 
        public_transit_time=public_transit_time, 
        travel_direction="TO_FACILITIES", 
        cutoffs=[i], 
        time_of_day=datetime.datetime(2024, 5, 15, 10, 0, 0),  # Wednesday 10:00 AM 
        geometry_at_overlaps="OVERLAP",
        facilities="UofGUC", 
    )

In [3]:
# ### Part3's service area analysis ### #
arcpy.env.workspace = fd_streets

# Import facilities (Great American Ballpark).
arcpy.conversion.FeatureClassToGeodatabase(
    Input_Features=r"C:\GEOG3440\GEOG3440_F24_Lab01_GuelphMarg\GuelphGroceries.shp", 
    Output_Geodatabase=fd_streets
)

# Set the public transit time.
network_data_source="TransitNetwork_ND"
modes = arcpy.na.GetTravelModes(network_data_source)
public_transit_time = modes["Public transit time"] 

# Run service area analysis for 10 and 20 minutes.
for i in [10, 20]:
    run_service_area_analysis(
        network_data_source=network_data_source, 
        layer_name=f"SA-toGroceries-Wed1400-{str(i)}min", 
        public_transit_time=public_transit_time, 
        travel_direction="TO_FACILITIES", 
        cutoffs=[i], 
        time_of_day=datetime.datetime(2024, 5, 15, 14, 0, 0),  # Wednesday 14:00 AM 
        geometry_at_overlaps="DISSOLVE",
        facilities="GuelphGroceries", 
    )